In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# 1. Complete 30 experimental data points from the spreadsheet (X, Y, Max VDC)
data = np.array([
[14, 8, -0.832],
[14, 5.5, 0.300],
[18, 5, -0.292],
[18, 10, -0.159],
[18, 14, -0.113],
[10, 14, -0.108],
[8, 12, -0.099],
[6, 10, -0.127],
[6, 6, -0.180],
[14, 13, -0.237],
[25, 5, -0.007],
[25, 10, -0.045],
[25, 15, -0.014],
[20, 18, -0.022],
[17, 18, -0.037],
[26, 18, -0.005],
[2, 18, -0.003],
[6, 12, -0.053],
[5, 3, -0.492],
[2, 4, -0.192],
[15, 3, -0.302],
[13, 3, -0.337],
[5, 16, -0.017],
[1, 19, -0.001],
[27, 10, -0.002],
[27, 10, -0.001], # Row 26/27 duplicate coordinates in sheet
[1, 1, 0.000],
[15, 9, -0.850],
[10, 8, -0.300],
[13, 11, -0.547]
])

x_measured, y_measured, v_measured = data[:, 0], data[:, 1], data[:, 2]

# 2. Recreate 26x20 lab grid (using 1cm spacing)
x_range = np.arange(0, 27, 1.0)
y_range = np.arange(0, 21, 1.0)
X, Y = np.meshgrid(x_range, y_range)

# 3. Interpolate data points across the whole grid space
V = griddata((x_measured, y_measured), v_measured, (X, Y), method='linear')
V_nearest = griddata((x_measured, y_measured), v_measured, (X, Y), method='nearest')
V[np.isnan(V)] = V_nearest[np.isnan(V)]

# 4. Calculate Experimental Electric Field vectors: E = -grad(V)
Gy, Gx = np.gradient(V, 1.0, 1.0)
Ex = -Gx * 100
Ey = -Gy * 100
E_magnitude = np.sqrt(Ex**2 + Ey**2)

# 5. Create color mapping array matching group requirements
colors = np.empty(X.shape, dtype=object)
colors[E_magnitude > 50] = '#e74c3c' # Red
colors[(E_magnitude >= 5) & (E_magnitude <= 50)] = '#f39c12' # Orange
colors[E_magnitude < 5] = '#95a5a6' # Gray

# Flatten arrays for the quiver tool
xf, yf = X.flatten(), Y.flatten()
exf, eyf = Ex.flatten(), Ey.flatten()
colors_f = colors.flatten()

# 6. Plotting the Real 30-Point Experimental Data Map
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
ax.set_facecolor('#ffffff')

# Plot experimental arrows
ax.quiver(xf, yf, exf, eyf, color=colors_f, scale=1200, width=0.004, headwidth=4, headlength=5, zorder=3)

# Plot actual physical points as dots
ax.scatter(x_measured, y_measured, color='blue', s=20, zorder=5, label='Measured Points')

# Formatting to match template
ax.set_xlim(0, 26)
ax.set_ylim(0, 20)
ax.set_xticks(np.arange(0, 27, 2))
ax.set_yticks(np.arange(0, 21, 2))
ax.set_xlabel('X-Coordinate (cm)', fontsize=11, fontweight='bold')
ax.set_ylabel('Y-Coordinate (cm)', fontsize=11, fontweight='bold')
ax.set_title('EXPERIMENTAL Electric Field Vector Map (30 Points)\nDerived from Real Laboratory Measurements', fontsize=12, fontweight='bold', pad=15)
ax.grid(True, which='both', color='#e0e0e0', linestyle='--', linewidth=0.5)
ax.set_aspect('equal')

# Clean legend
ax.plot([], [], marker='>', color='none', markerfacecolor='#e74c3c', markeredgecolor='#e74c3c', markersize=10, label='Strong Field (>50 V/m)')
ax.plot([], [], marker='>', color='none', markerfacecolor='#f39c12', markeredgecolor='#f39c12', markersize=10, label='Moderate Field (5-50 V/m)')
ax.plot([], [], marker='>', color='none', markerfacecolor='#95a5a6', markeredgecolor='#95a5a6', markersize=10, label='Weak Field (<5 V/m)')
ax.legend(loc='upper right', framealpha=0.9)

plt.savefig('experimental_30point_vectors.png', bbox_inches='tight', dpi=300)
plt.show()